# Little Lemon booking system client

Connects Python to the Little Lemon MySQL database with MySQL Connector/Python.

## Working environment

In [1]:
import sys

print(sys.version)

3.14.10 (tags/v3.14.10:b703b2c, Oct 25 2026, 17:36:52) [MSC v.1944 64 bit (AMD64)]


In [2]:
!pip install mysql-connector-python

In [3]:
import mysql.connector as connector

In [4]:
# A database user that exists only for this project, on the local server.
connection = connector.connect(user="littlelemon", password="lemon-capstone")

print("Connected:", connection.is_connected())

Connected: True


## Query functions

Connect to the Little Lemon database and open a cursor.

In [5]:
connection = connector.connect(
    user="littlelemon",
    password="lemon-capstone",
    db="LittleLemonDB",
)
cursor = connection.cursor()

print("Connected to:", connection.database)

Connected to: littlelemondb


Show all tables in the database.

In [6]:
show_tables_query = "SHOW tables"
cursor.execute(show_tables_query)

results = cursor.fetchall()
print(results)

[('bookings',), ('customers',), ('menuitems',), ('menus',), ('orderdeliverystatus',), ('orders',), ('ordersview',), ('staff',)]


Full name, contact details and bill amount of every customer who spent more than $60 on an order.

In [7]:
join_query = """
SELECT Customers.FullName, Customers.ContactNumber, Customers.Email, Orders.TotalCost
FROM Customers
INNER JOIN Orders ON Customers.CustomerID = Orders.CustomerID
WHERE Orders.TotalCost > 60
ORDER BY Orders.TotalCost
"""
cursor.execute(join_query)

results = cursor.fetchall()
print(cursor.column_names)
for row in results:
    print(row)

('FullName', 'ContactNumber', 'Email', 'TotalCost')
('Vanessa McCarthy', '0757536378', 'vanessa.mccarthy@example.com', Decimal('90.00'))
('Marcos Romero', '0757536379', 'marcos.romero@example.com', Decimal('155.00'))
('Diana Pinto', '0757536374', 'diana.pinto@example.com', Decimal('175.00'))
('Vanessa McCarthy', '0757536378', 'vanessa.mccarthy@example.com', Decimal('250.00'))


## Stored procedures

Call the stored procedures of the database from Python.

In [8]:
def call_procedure(name, args=()):
    """Call a stored procedure and return the rows it selects."""
    cursor.callproc(name, args)
    rows = []
    for result in cursor.stored_results():
        rows.extend(result.fetchall())
    return rows

The largest quantity in any order.

In [10]:
print(call_procedure("GetMaxQuantity"))

[(5,)]


C:\Users\joris\AppData\Local\Temp\3\ipykernel_9152\614344863.py:5: DeprecationWarning: Call to deprecated function stored_results. Reason: The property counterpart 'stored_results' will be added in a future release, and this method will be removed.
  for result in cursor.stored_results():


Is a table free on a date? Table 5 is booked on 2022-10-10, table 6 is not.

In [11]:
print(call_procedure("ManageBooking", ("2022-10-10", 5)))
print(call_procedure("ManageBooking", ("2022-10-10", 6)))

[('Table 5 is already booked',)]
[('Table 6 is available',)]


C:\Users\joris\AppData\Local\Temp\3\ipykernel_9152\614344863.py:5: DeprecationWarning: Call to deprecated function stored_results. Reason: The property counterpart 'stored_results' will be added in a future release, and this method will be removed.
  for result in cursor.stored_results():


In [12]:
cursor.close()
connection.close()

print("Connection closed")

Connection closed
